# Fabric CI/CD Initial Deployment
Gebruik dit notebook om de repository te clonen en direct naar een Fabric workspace te deployen met `config.yml`.

In [ ]:
%pip install -q fabric-cicd azure-identity

In [ ]:
import os
import tempfile
import subprocess
from fabric_cicd import deploy_with_config

workspace_id = "14cffcef-f00f-4704-85a8-9591654ba168"
environment = "default"
repo_url = "https://github.com/nb-tosch/FabricTest.git"
repo_ref = "main"

with tempfile.TemporaryDirectory(prefix="cloned_repo_") as temp_dir:
    print(f"Cloning repo to: {temp_dir}")
    subprocess.run(
        ["git", "clone", "--branch", repo_ref, "--single-branch", repo_url, temp_dir],
        check=True
    )

    config_file_path = os.path.join(temp_dir, "config.yml")

    # Override runtime values without changing files in the repo
    config_override = {
        "core": {
            "workspace_id": {environment: workspace_id},
            "repository_directory": "."
        },
        "publish": {"skip": {environment: False}},
        "unpublish": {"skip": {environment: True}}
    }

    result = deploy_with_config(
        config_file_path=config_file_path,
        environment=environment,
        config_override=config_override
    )

    print(f"Deployment status: {result.status}")
    print(f"Deployment message: {result.message}")